# Phase 2 — Predictions Table + OpenAI Insights

Two jobs, auto-selected from the data:

1. **Insights only** — if `predictions` is already ahead of Gold (e.g. 2026 Week 1 from the slate builder, before any games), load that partition and attach GPT-5.6 Terra writeups. Does **not** retrain. The UI reads `insight` from this table.
2. **Predict + insights** — if the latest Gold week matches or leads predictions (a completed week), walk-forward project that week from `player_weeks`, then generate insights. Same fold as `walk_forward_model.ipynb`.

**Setup:** OpenAI key in `insights/.env` (`OPENAI_API_KEY`) or Databricks secret scope `openai-creds` / `api-key`. Pin `gpt-5.6-terra` — the `gpt-5.6` alias routes to Sol. Missing key → template insights so the write still completes.

No git push is required for the dashboard. After this notebook writes, restart the API (or wait ~1 hour) and refresh the UI.

In [0]:
# Install dependencies
%pip install xgboost scikit-learn openai python-dotenv

## Load the target week

Compares the newest `(season, week)` in `fantasy_football.gold.predictions` to Gold `player_weeks`. A forward slate (Week 1 before kickoff) skips training and only prepares rows for RAG. A completed week retrains and overwrites that partition, then continues to insights.

In [0]:
import os
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

# Load gold + existing predictions. If predictions already has a *future* slate
# (newer than Gold — 2026 Week 1 before kickoff), only attach insights later.
# Otherwise walk-forward project the latest completed Gold week.
if 'gold_df' not in globals():
    gold_df = spark.table("fantasy_football.gold.player_weeks").toPandas()

gold_df = gold_df[gold_df['week'] <= 17].copy()
TARGET = 'fantasy_points_ppr'

gold_season = int(gold_df['season'].max())
gold_week = int(gold_df.loc[gold_df['season'] == gold_season, 'week'].max())

pred_tbl = spark.table("fantasy_football.gold.predictions").toPandas()
pred_season = int(pred_tbl['season'].max())
pred_week = int(pred_tbl.loc[pred_tbl['season'] == pred_season, 'week'].max())

forward_slate = (pred_season, pred_week) > (gold_season, gold_week)

if forward_slate:
    PREDICT_SEASON, PREDICT_WEEK = pred_season, pred_week
    predictions_df = pred_tbl[
        (pred_tbl['season'] == PREDICT_SEASON) & (pred_tbl['week'] == PREDICT_WEEK)
    ].copy()
    predictions_df = predictions_df.sort_values('projected_ppr', ascending=False).reset_index(drop=True)
    print(
        f"Forward slate {PREDICT_SEASON} week {PREDICT_WEEK}: "
        f"{len(predictions_df)} rows already in predictions — skipping retrain."
    )
else:
    PREDICT_SEASON, PREDICT_WEEK = gold_season, gold_week
    eval_meta = gold_df[['player_id', 'player_name', 'recent_team', 'position',
                         'season', 'week', 'opponent']].copy()

    df = gold_df.copy()
    identifier_cols = ['player_id', 'player_name', 'recent_team', 'opponent', 'starting_qb_id', 'gameday']
    df = df.drop(columns=[c for c in identifier_cols if c in df.columns])

    same_week_outcome_cols = [
        'pass_attempts', 'completions', 'passing_yards', 'passing_tds', 'interceptions',
        'rush_attempts', 'rushing_yards', 'rushing_tds',
        'targets', 'receptions', 'receiving_yards', 'receiving_tds',
        'player_opportunities', 'team_total_opportunities', 'opportunity_share',
        'hvt_carries', 'hvt_targets', 'total_hvts',
        'team_pass_attempts', 'target_share',
        'player_air_yards', 'team_air_yards', 'air_yards_share',
        'wopr', 'snap_share',
    ]
    df = df.drop(columns=[c for c in same_week_outcome_cols if c in df.columns])
    df['position'] = eval_meta['position']
    df = pd.get_dummies(df.fillna(0), columns=['position'], prefix='pos')
    feature_cols = [c for c in df.columns if c != TARGET]

    in_season = eval_meta['season'] == PREDICT_SEASON
    test_mask = in_season & (eval_meta['week'] == PREDICT_WEEK)
    val_mask = in_season & (eval_meta['week'] == PREDICT_WEEK - 1)
    train_mask = ~in_season | (eval_meta['week'] <= PREDICT_WEEK - 2)

    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8,
        early_stopping_rounds=20, random_state=42, n_jobs=-1,
    )
    model.fit(
        df.loc[train_mask, feature_cols], df.loc[train_mask, TARGET],
        eval_set=[(df.loc[val_mask, feature_cols], df.loc[val_mask, TARGET])],
        verbose=False,
    )
    preds = np.clip(model.predict(df.loc[test_mask, feature_cols]), 0, None)

    context_cols = ['implied_total', 'team_spread', 'team_win_prob', 'is_home',
                    'temp', 'wind', 'is_bad_weather', 'is_dome',
                    'fantasy_points_3wk_avg', 'depth_chart_rank',
                    'opp_def_ppg_allowed', 'prev_season_ppg']
    predictions_df = eval_meta.loc[test_mask].reset_index(drop=True)
    predictions_df['projected_ppr'] = preds.astype(float).round(1)
    predictions_df['actual_ppr'] = df.loc[test_mask, TARGET].astype(float).round(1).values
    predictions_df = pd.concat(
        [predictions_df, df.loc[test_mask, context_cols].round(2).reset_index(drop=True)], axis=1
    )
    predictions_df = predictions_df.sort_values('projected_ppr', ascending=False).reset_index(drop=True)

    spark.createDataFrame(predictions_df).write \
        .format("delta") \
        .mode("overwrite") \
        .option("replaceWhere", f"season = {PREDICT_SEASON} AND week = {PREDICT_WEEK}") \
        .saveAsTable("fantasy_football.gold.predictions")
    print(f"Projected {PREDICT_SEASON} week {PREDICT_WEEK}: {len(predictions_df)} players")

show_cols = [c for c in ['player_name', 'recent_team', 'position', 'opponent',
                         'projected_ppr', 'actual_ppr', 'implied_total']
             if c in predictions_df.columns]
display(predictions_df.head(10)[show_cols])

## RAG Flow: Retrieve → Augment → Generate

Writes 2–3 sentence insights for the top 15 projected players on the week loaded above. The prompt uses only columns already on the predictions row (projection, Vegas, depth, prev-season PPG, venue). Week 1 rows have no in-season form or opponent-defense splits yet — the model is told that explicitly instead of treating zeros as real stats.

In [0]:
# RAG: RETRIEVE stats -> AUGMENT prompt -> GENERATE insight via OpenAI API
from dotenv import load_dotenv

load_dotenv()

OPENAI_MODEL = 'gpt-5.6-terra'

try:
    OPENAI_API_KEY = dbutils.secrets.get(scope="openai-creds", key="api-key")
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY')
    if not OPENAI_API_KEY:
        print("Warning: OpenAI API key not found in .env, secrets, or environment.")
        print("Will fall back to template insights.")

OPENAI_API_BASE = os.environ.get('OPENAI_API_BASE')

N_INSIGHTS = 15


def _num(row, col, default=0.0):
    v = row[col] if col in row.index else default
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return default
    return float(v)


def build_prompt(row):
    """AUGMENT step: inject retrieved model output + context into the prompt."""
    is_home = bool(_num(row, 'is_home'))
    is_dome = bool(_num(row, 'is_dome'))
    is_bad = bool(_num(row, 'is_bad_weather'))
    venue = 'at home' if is_home else 'on the road'
    form = _num(row, 'fantasy_points_3wk_avg')
    opp_def = _num(row, 'opp_def_ppg_allowed')
    form_line = (
        f"Recent form (3-week avg): {form} PPR points"
        if form
        else "Recent form (3-week avg): not available yet (no games played this season)"
    )
    matchup_line = (
        f"Opponent defense allows {opp_def} PPR points/game to {row['position']}s"
        if opp_def
        else f"Opponent {row['position']} defense allowed: not available yet (preseason / week 1)"
    )
    if is_dome:
        weather = 'indoors (dome)'
    else:
        weather = f"{_num(row, 'temp'):.0f}F, {_num(row, 'wind'):.0f} mph wind"
        if is_bad:
            weather += ' — bad weather game'
    depth = int(_num(row, 'depth_chart_rank', 9))
    return f"""You are a fantasy football analyst. Using ONLY the data below, write a 2-3 sentence
insight for this player's upcoming game. Mention the projection, one supporting factor,
and one risk factor. Do not invent injuries or news not present in the data.
If recent form or opponent defense is listed as not available, do not treat it as zero skill.

Player: {row['player_name']} ({row['position']}, {row['recent_team']})
Opponent: {row['opponent']} ({venue})
Model projection: {row['projected_ppr']} PPR points
{form_line}
Previous season average: {_num(row, 'prev_season_ppg')} PPR points
Vegas implied team total: {_num(row, 'implied_total')} | spread: {_num(row, 'team_spread'):+.1f} | win prob: {_num(row, 'team_win_prob'):.0%}
{matchup_line}
Depth chart rank: {depth}
Weather: {weather}"""


def template_insight(row):
    """Offline fallback so the pipeline runs end-to-end without an API key."""
    spread = _num(row, 'team_spread')
    lean = 'favorable' if spread > 0 else 'tough'
    form = _num(row, 'fantasy_points_3wk_avg')
    form_bit = (
        f"and he has averaged {form:.1f} points over the last three weeks."
        if form
        else "with no in-season form yet (week 1)."
    )
    parts = [
        f"{row['player_name']} projects for {row['projected_ppr']} PPR points against {row['opponent']}.",
        f"Vegas implies a {_num(row, 'implied_total'):.1f}-point team total in a {lean} game script, {form_bit}",
    ]
    opp_def = _num(row, 'opp_def_ppg_allowed')
    if _num(row, 'is_bad_weather'):
        parts.append("Bad weather is a downside risk for this game.")
    elif opp_def and opp_def < 15:
        parts.append(
            f"Risk: {row['opponent']} has been stingy against {row['position']}s "
            f"({opp_def:.1f} PPR pts/game allowed)."
        )
    elif _num(row, 'prev_season_ppg') == 0:
        parts.append("Risk: no prior-season scoring history (rookie or unranked).")
    return ' '.join(parts)


def make_llm():
    if not OPENAI_API_KEY:
        print("No OpenAI API key — using offline template insights.")
        return 'offline_template', None

    try:
        from openai import OpenAI

        client_kwargs = {'api_key': OPENAI_API_KEY}
        if OPENAI_API_BASE:
            client_kwargs['base_url'] = OPENAI_API_BASE
            print(f"Using custom API endpoint: {OPENAI_API_BASE}")

        client = OpenAI(**client_kwargs)
        client.responses.create(
            model=OPENAI_MODEL, input='hi', max_output_tokens=16, store=False,
        )

        def gen(prompt):
            resp = client.responses.create(
                model=OPENAI_MODEL,
                input=prompt,
                reasoning={'effort': 'low'},
                max_output_tokens=500,
                store=False,
            )
            return resp.output_text.strip()

        return OPENAI_MODEL, gen
    except Exception as e:
        print(f"OpenAI API unavailable ({type(e).__name__}) — using offline template insights.")
        return 'offline_template', None


def generate_insight(gen_fn, row):
    return gen_fn(build_prompt(row)) if gen_fn else template_insight(row)


predictions_df = predictions_df.drop(columns=['insight', 'insight_source'], errors='ignore')

llm_name, gen_fn = make_llm()
print(f"Insight generator: {llm_name}")
top = predictions_df.head(N_INSIGHTS).copy()
top['insight'] = [generate_insight(gen_fn, row) for _, row in top.iterrows()]
top['insight_source'] = llm_name

final = predictions_df.merge(
    top[['player_id', 'insight', 'insight_source']], on='player_id', how='left'
)
final_spark = spark.createDataFrame(final)

final_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("replaceWhere", f"season = {PREDICT_SEASON} AND week = {PREDICT_WEEK}") \
    .option("mergeSchema", "true") \
    .saveAsTable("fantasy_football.gold.predictions")

print(f"✓ Saved {PREDICT_SEASON} week {PREDICT_WEEK} predictions + insights to fantasy_football.gold.predictions\n")

for _, row in top.head(5).iterrows():
    actual = row['actual_ppr'] if 'actual_ppr' in row.index else None
    actual_txt = '—' if actual is None or (isinstance(actual, float) and pd.isna(actual)) else actual
    print(f"--- {row['player_name']} ({row['position']}, {row['recent_team']}) "
          f"proj {row['projected_ppr']} | actual {actual_txt} ---")
    print(row['insight'], '\n')